# E07: Testing y Calidad de Código

> **Nivel:** Intermedio | **Python 3.12+**

El software sin tests es como un puente sin inspección: puede sostenerse hoy, pero no tienes evidencia de que resistirá mañana. Este notebook te enseña a escribir **pruebas automatizadas** que protegen, documentan y guían tu código.

Trabajaremos con los **dos frameworks** esenciales:

- **unittest** → viene en la biblioteca estándar, **siempre funciona** (no requiere instalación).
- **pytest** → el estándar moderno de la industria, más conciso y potente.

Al final sabrás cuándo, cómo y qué testear, y cómo aplicar **TDD** (Desarrollo Guiado por Tests) y **calidad de código** en tus proyectos.

## Objetivos

Al terminar este notebook podrás:

1. **Explicar** el valor de los tests y decidir *cuándo* (y cuándo no) testear, con base en el costo/beneficio.
2. **Escribir** pruebas con `unittest` (stdlib): clases `TestCase`, `setUp`/`tearDown` y los `assert*` clave.
3. **Escribir** pruebas con `pytest`: funciones `test_*`, `assert` simple, `fixtures`, `parametrize` y `markers`.
4. **Aplicar** el ciclo **TDD** (Red → Green → Refactor) paso a paso sobre una función real.
5. **Usar** `unittest.mock` para aislar dependencias externas (red, base de datos).
6. **Interpretar** la cobertura de código y aplicar principios de calidad (DRY, KISS, SOLID) para diseñar código comprobable.

## Analogía: la red de seguridad del escalador

Imagina a un escalador profesional. Nunca sube sin su **red de seguridad** o su **arnés**. Cada `assert` de tus tests funciona igual: **atrapa tus errores antes de que llegues al suelo** (producción).

```
         \  clifff  /
           .     .
          __________________
         |  TU CÓDIGO (TU \  |  <- el escalador
         |  ROCKCLIMBER)   \ |
         |__________________\|
                |  |  |
           RED DE SEGURIDAD      <- los TESTS
        +------------------------+
        |  Te atrapan si caes     |
        +------------------------+

    Sin tests  : caes al vacio directamente
    Con tests  : caes... y te sostiene la red, luego lo arreglas
```

Por otro lado, **TDD (Desarrollo Guiado por Tests)** es como firmar un **contrato** antes de construir. Primero escribes *qué* debe cumplir la función (el test), y *después* implementas *cómo* lo cumple. El test es el contrato que documenta el comportamiento esperado para siempre.

## 1. ¿Por qué testear?

Testear no es 'perder tiempo'. Es **inversión**. Un test bien escrito se amortiza en el primer fallo que detecta antes de llegar a producción.

### Beneficios

| Beneficio | Qué te da |
|-----------|-----------|
| **Regresión** | Detectas errores que introduces al agregar código nuevo |
| **Documentación viva** | El test muestra *cómo* se usa la función y qué se espera |
| **Confianza** | Refactorizas sin miedo: los tests confirman que nada se rompió |
| **Diseño** | Escribir tests te obliga a hacer código más desacoplado y reutilizable |
| **Onboarding** | Los nuevos miembros entienden el sistema leyendo los tests |

### Cuándo SÍ testear

- Lógica de negocio con reglas (impuestos, descuentos, validaciones).
- Algoritmos con casos límite (ordenamientos, búsquedas, parsing).
- APIs y funciones que otros módulos usan.
- Código que corregiste una vez por un bug → nevera contra regresión.

### Cuándo NO es necesario

- **Prototipos desechables** que borrarás mañana.
- Código que es pura envoltura de una librería ya testada (no testes a pandas por segunda vez).
- Scripts de una sola ejecución sin lógica reutilizable.

### Costo / beneficio

```
Costo del bug
    ^
    |                   \  (bug en producción = caro)
    |                      \
    |   +--------+              \
    |   | TESTS  |                 \
    |   | aquí   |                    \
    |   | (bajo) |                      \
    +---+--------+-----------------------\-------->  Tiempo
      escritura     integracion       produccion

  Detectar el error TEMPRANO (en la escritura del codigo)
  es mucho mas barato que en produccion
```

**Regla práctica:** si el código es **reutilizable** o **contiene lógica condicional**, vale la pena testearlo.

In [ ]:
# Empecemos con una funcion sencilla que luego probaremos.
# Usamos type hints (Python 3.12+) y un docstring tipo contrato.

def calcular_impuesto(precio: float, tasa: float = 0.16) -> float:
    """Calcula el impuesto (IVA) sobre un precio.

    precio: monto base, debe ser >= 0.
    tasa:   fraccion impositiva (por defecto 0.16 = 16%).

    >>> calcular_impuesto(100)
    16.0
    """
    if precio < 0:
        raise ValueError("el precio no puede ser negativo")
    return round(precio * tasa, 2)


print(calcular_impuesto(100))      # 16.0
print(calcular_impuesto(200, 0.10)) # 20.0

## 2. unittest (biblioteca estándar)

`unittest` viene **incluido en Python**, así que *siempre* funciona, sin instalar nada. Es el punto de partida ideal cuando trabajas en entornos controlados o sin internet.

### Estructura de un archivo de test

```
tests/
└── test_calculadora.py
    ├── import unittest
    ├── import (tu modulo con el codigo)
    ├── class TestCalculadora(unittest.TestCase):   <- agrupa casos
    │     ├── def setUp(self):            <- se ejecuta antes de cada test
    │     ├── def tearDown(self):         <- se ejecuta despues de cada test
    │     └── def test_impuesto(self):    <- cada metodo test_* es un caso
    │           self.assertEqual(...)
    └── if __name__ == "__main__":
         unittest.main()
```

### Aserciones clave

| Método | Verifica que... | Alternativa `assert` |
|--------|-----------------|----------------------|
| `assertEqual(a, b)` | `a == b` | `assert a == b` |
| `assertTrue(x)` | `x` es verdadero | `assert x` |
| `assertFalse(x)` | `x` es falso | `assert not x` |
| `assertAlmostEqual(a, b)` | `a ≈ b` (flotantes) | `abs(a-b) < tol` |
| `assertIsNone(x)` | `x` es `None` | `assert x is None` |
| `assertIn(a, b)` | `a` está en `b` | `assert a in b` |
| `assertRaises(Exc)` | se lanza `Exc` | `pytest.raises(...)` |
| `assertRaisesRegex(Exc, rx)` | se lanza `Exc` con mensaje `rx` | `pytest.raises(..., match=...)` |

### Cómo correr los tests

```bash
# Opcion A: desde la terminal
python -m unittest discover tests -v

# Opcion B: ejecutando el archivo directamente
python tests/test_calculadora.py
```

In [ ]:
# Definimos el MODULO bajo test en el notebook.
# En un proyecto real, calcular_impuesto viviria en calculadora.py

def calcular_impuesto(precio: float, tasa: float = 0.16) -> float:
    """Calcula el impuesto (IVA) sobre un precio."""
    if precio < 0:
        raise ValueError("el precio no puede ser negativo")
    return round(precio * tasa, 2)

In [ ]:
# Ahora la CLASE de tests con unittest.
# Observa: cada metodo que empieza por 'test_' sera ejecutado.

import unittest


class TestCalcularImpuesto(unittest.TestCase):
    """Pruebas para calcular_impuesto usando unittest."""

    def setUp(self):
        """Se ejecuta antes de CADA test. Aqui montamos el 'andamio'."""
        self.tasa_default = 0.16

    def tearDown(self):
        """Se ejecuta despues de CADA test. Aqui limpias recursos."""
        pass

    def test_impuesto_default(self):
        self.assertEqual(calcular_impuesto(100), 16.0)

    def test_impuesto_con_tasa_personalizada(self):
        self.assertEqual(calcular_impuesto(200, 0.10), 20.0)

    def test_precio_cero(self):
        self.assertEqual(calcular_impuesto(0), 0.0)

    def test_flotantes_usar_almost(self):
        # Para flotantes NO uses assertEqual, usa assertAlmostEqual
        self.assertAlmostEqual(calcular_impuesto(100.0, 0.1), 10.0000001)

    def test_precio_negativo_lanza_error(self):
        # assertRaises como gestor de contexto: verifica que se LANZA la excepcion
        with self.assertRaises(ValueError):
            calcular_impuesto(-10)

    def test_error_con_mensaje(self):
        with self.assertRaisesRegex(ValueError, "negativo"):
            calcular_impuesto(-5)


# Esto permite correr el archivo directamente:  python test_calculadora.py
if __name__ == "__main__":
    unittest.main(verbosity=2)

### Cómo ejecutarlo en el notebook

En un notebook podemos correr los tests de una clase con `unittest.TextTestRunner`. Esto nos **demuestra** que la lógica funciona, sin necesidad de archivos externos ni instalar nada.

In [ ]:
# Ejecutamos la suite de tests definida arriba y vemos el reporte.

import unittest

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCalcularImpuesto)
resultado = unittest.TextTestRunner(verbosity=2).run(suite)

print(f"\n-> {resultado.testsRun} tests, {len(resultado.failures)} fallos, {len(resultado.errors)} errores")

## 3. pytest (el estándar moderno)

Si `unittest` es el martillo clásico, **pytest** es la caja de herramientas completa. Es el framework **más usado en la industria** por su **concisión**, su **ecosistema de plugins** y su **soporte de `fixtures`**.

### Diferencias clave frente a unittest

| unittest | pytest |
|----------|--------|
| Clases `TestCase` | Funciones `test_*` (más simple) |
| Métodos `self.assertEqual` | `assert` de Python puro |
| `setUp`/`tearDown` | `fixtures` con `yield` |
| Sin parametrizar fácil | `@pytest.mark.parametrize` (múltiples casos) |
| `skip`/`expectedFailure` | `@pytest.mark.skip` / `@pytest.mark.xfail` |
| Reporte básico | Reporte rico con colores y diagnóstico |

> **Instalación:** `pip install pytest` (opcional). Si no tienes pytest, **unittest sigue siendo 100% funcional**. Este notebook te muestra ambos por esa razón.

### Los 3 superpoderes de pytest

1. **`assert` simple**: usas Python puro, no métodos especiales.
2. **`fixtures`**: preparan/limpian contexto de forma reutilizable (con `yield` separas la preparación de la limpieza).
3. **`parametrize`**: ejecuta el mismo test con *muchos* casos de entrada en un solo bloque.

In [ ]:
# FUNCIONES de test al estilo pytest.
# No requieren clases: basta que el nombre empiece por 'test_'.
# Usamos 'assert' de Python puro, sin self.assertEqual.

import io
import contextlib

# Reusamos la funcion ya definida en celdas anteriores

def test_impuesto_default():
    assert calcular_impuesto(100) == 16.0


def test_impuesto_con_tasa():
    assert calcular_impuesto(200, 0.10) == 20.0


def test_precio_negativo_lanza_error():
    import pytest
    with pytest.raises(ValueError):
        calcular_impuesto(-10)


def test_muchos_casos():
    # Un solo test puede validar varios escenarios en bucle
    casos = [(100, 16.0), (250, 40.0), (0, 0.0), (1, 0.16)]
    for precio, esperado in casos:
        assert calcular_impuesto(precio) == esperado

In [ ]:
# CORRER pytest desde el notebook con '!pytest' (magico de shell).
# Solo funciona si pytest esta instalado; si no esta, capturamos el error.

import subprocess

try:
    resultado = subprocess.run(
        ["pytest", "-q", "--no-header"],
        capture_output=True, text=True, timeout=60,
    )
    print(resultado.stdout)
    print(resultado.stderr)
except FileNotFoundError:
    print("pytest NO esta instalado en este entorno.")
    print("Instala con:  pip install pytest")
    print("Mientras tanto, unittest (stdlib) ya cubrió estas pruebas.")

### Fixtures: preparación reutilizable

Un **fixture** es una función que *prepara* un contexto (datos, conexiones, objetos) que tus tests necesitan. Es la evolución más elegante de `setUp`/`tearDown`:

- Todo lo que está **antes de `yield`** se ejecuta como 'preparación' (setup).
- Todo lo que está **después de `yield`** se ejecuta como 'limpieza' (teardown).
- Puedes **reutilizar** el mismo fixture en varios tests u otros fixtures.

In [ ]:
# Ejemplo de FIXTURE con yield (separando setup y teardown).

import tempfile
import os

def archivo_temporal():
    """Fixture: crea un archivo temporal y lo limpia al terminar."""
    fd, ruta = tempfile.mkstemp(suffix=".txt")
    print("[setup] archivo creado en", ruta)
    yield ruta          # -> aqui se inyecta el valor en el test
    os.close(fd)
    os.remove(ruta)     # <- limpieza (teardown)
    print("[teardown] archivo eliminado")


def test_fixture_con_yield():
    gen = archivo_temporal()
    ruta = next(gen)               # ejecuta hasta el yield (setup)
    assert os.path.exists(ruta)
    print("-> test usando ruta:", ruta)
    try:
        next(gen)                  # provoca GeneratorExit -> teardown
    except StopIteration:
        pass
    print("-> test terminado; el archivo fue limpiado")


test_fixture_con_yield()

In [ ]:
# PARAMETRIZE: un test ejecutado con muchos casos.
# Es la forma limpia de cubrir la matriz de combinaciones.

import pytest

# Lista de (precio, tasa, esperado)
CASOS_IMPUESTO = [
    (100, 0.16, 16.0),
    (200, 0.10, 20.0),
    (0, 0.16, 0.0),
    (50, 0.00, 0.0),
    (333, 0.16, 53.28),
]

@pytest.mark.parametrize("precio,tasa,esperado", CASOS_IMPUESTO)
def test_impuesto_parametrizado(precio, tasa, esperado):
    # Cada caso se trata como una ejecucion INDEPENDIENTE del test
    assert calcular_impuesto(precio, tasa) == esperado


print("Casos definidos. pytest ejecutaria test_impuesto_parametrizado")
print("5 veces (una por cada tupla de CASOS_IMPUESTO).")
for precio, tasa, esperado in CASOS_IMPUESTO:
    resultado = calcular_impuesto(precio, tasa)
    assert resultado == esperado, f"fallo en {precio},{tasa} -> {resultado}"
print("Todos los casos parametrizados pasan.")

In [ ]:
# MARKERS: skip y xfail controlan casos especiales.

import sys

@pytest.mark.skip(reason="Funcionalidad aun no implementada")
def test_impuesto_dolarizado():
    # Este test se OMITE sin fallar la suite
    assert False  # nunca se ejecuta


@pytest.mark.skipif(sys.version_info < (3, 10), reason="Requiere Python 3.10+")
def test_solo_python_reciente():
    assert 1 + 1 == 2


@pytest.mark.xfail(reason="Bug conocido: redondeo en centavos", strict=False)
def test_redondeo_conocido():
    # Esperamos que FALLE (xfail); si de repente pasa, pytest lo avisa (xpassed)
    assert calcular_impuesto(0.1, 0.16) == 0.02  # 0.1*0.16=0.016, al cuadrado -> 0.02


print("Markers definidos. En pytest:")
print("  - skip   -> el test se omite")
print("  - skipif -> se omite si la condicion es verdadera")
print("  - xfail  -> se espera que falle (fallo esperado no quiebra la suite")

In [ ]:
# IMPORTANTE: si pytest NO está instalado, usar pytest.
# en el notebook no rompe, pero no puede correr. Demostramos con unittest
# como alternativa 100% funcional usando la MISMA logica.

import unittest

try:
    import pytest  # noqa: F401
    PYTEST_OK = True
except ImportError:
    PYTEST_OK = False

print("pytest disponible:", PYTEST_OK)
print("Si no está disponible, unittest cubre exactamente los mismos casos.")

### Cómo correr pytest (resumen)

```bash
# En terminal / CI
pytest                          # busca archivos test_*.py y funciones test_*
pytest -q                      # modo silencioso
pytest -v                      # verboso: muestra cada test
pytest test_calculadora.py     # un archivo especifico
pytest tests/ -k impuesto      # filtra por nombre

# Desde el notebook
!pytest -q                     # via magic de shell
```

## 4. TDD: Red → Green → Refactor

**TDD** (Test Driven Development) invierte el orden natural: **los tests se escriben ANTES que el código de producción**. Es un contrato que dicta el comportamiento esperado.

### El ciclo

```
      +------------------------------------------+
      |                                          |
      v                                          |
  +---------+     escribes test     +---------+  |
  |  RED    | --------------------> | test    |  |
  |  (rojo) |                        | FALLA   |  |
  +---------+                        +---------+  |
        ^                             |           |
        |  (refactor sin             v            |
        |   romper verde)        +---------+      |
        |                        |  IMPL   |      |
        +-------- GREEN <--------| codigo  |      |
        |          (verde)       +---------+      |
        |                  escribes el minimo     |
        |                  para que pase          |
        |                                         |
        +-----------------------------------------+  (repetir)

1. RED    : escribes un test que FALLA   (define el comportamiento)
2. GREEN  : escribes el minimo codigo    (para que el test pase)
3. REFACTOR: limpias el codigo           (sin romper el verde)
```

### Reglas de oro

- Solo escribes **un** test a la vez.
- Escribes el **mínimo** código que haga pasar ese test (no más).
- Refactorizas **solo cuando todo está en verde**.
- El test que no ha visto fracasar **no cuenta** (no sabes si prueba algo).

In [ ]:
# PASO 1 (RED): Escribimos el TEST primero, ANTES de implementar la funcion.
# La funcion calcular_descuento NO existe todavia -> el test debe FALLAR.

import unittest


class TestCalcularDescuento(unittest.TestCase):
    def test_descuento_basico(self):
        self.assertEqual(calcular_descuento(100, 0.10), 10.0)

    def test_descuento_cero(self):
        self.assertEqual(calcular_descuento(100, 0.0), 0.0)

    def test_descuento_total(self):
        self.assertEqual(calcular_descuento(100, 1.0), 100.0)


try:
    calcular_descuento  # la funcion aun no existe en el namespace
    raise RuntimeError("la funcion ya existia; red no se demostro")
except NameError:
    print("RED OK: la funcion 'calcular_descuento' NO existe aun.")
    print("El test, tal como esta definido, FALLARIA (NameError).")

In [ ]:
# PASO 2 (GREEN): Ahora SI implementamos la funcion, el minimo necesario
# para que el test definido arriba pase.

def calcular_descuento(precio: float, porcentaje: float) -> float:
    """Calcula el descuento sobre un precio."""
    if not (0 <= porcentaje <= 1):
        raise ValueError("el porcentaje debe estar entre 0 y 1")
    return round(precio * porcentaje, 2)


# Verificamos que ahora TODOS los tests pasan (GREEN)
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCalcularDescuento)
resultado = unittest.TextTestRunner(verbosity=2).run(suite)
print(f"\nGREEN: {resultado.testsRun} tests, {len(resultado.failures)} fallos")

In [ ]:
# PASO 3 (REFACTOR): Limpiamos el codigo SIN cambiar el comportamiento.
# Extraemos la validacion a una funcion auxiliar reutilizable (DRY).

def _validar_porcentaje(porcentaje: float) -> None:
    if not (0 <= porcentaje <= 1):
        raise ValueError("el porcentaje debe estar entre 0 y 1")


def calcular_descuento(precio: float, porcentaje: float) -> float:
    """Calcula el descuento sobre un precio (refactorizado)."""
    _validar_porcentaje(porcentaje)
    return round(precio * porcentaje, 2)


def calcular_impuesto(precio: float, tasa: float = 0.16) -> float:
    """Calcula el impuesto; reusa la validacion de porcentaje."""
    _validar_porcentaje(tasa)
    if precio < 0:
        raise ValueError("el precio no puede ser negativo")
    return round(precio * tasa, 2)


# Refactor sin romper el verde: re-corremos la suite completa
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCalcularDescuento)
resultado = unittest.TextTestRunner(verbosity=1).run(suite)
print(f"\nREFACTOR OK: {resultado.testsRun} tests siguen pasando")

## 5. Mocks y dobles de prueba

A veces tu código depende de **recursos externos**: una API de red, una base de datos, un reloj, un archivo. No quieres que los tests dependan de esos recursos (lentos, inestables, caros o imposibles en CI).

Ahí entran los **dobles de prueba**: sustitutos controlados que imitan el comportamiento real.

| Doble | Qué hace |
|-------|----------|
| **Mock** | Objeto falso que registra llamadas y responde con valores configurados |
| **Stub** | Devuelve valores predefinidos (sin registrar llamadas) |
| **Fake** | Implementación ligera real (p.ej. BD en memoria) |
| **Spy** | Envuelve el objeto real y registra interacciones |

### `unittest.mock`

Python incluye `unittest.mock` con la clase `Mock` y los decoradores `patch` / `patch.object`.

- `Mock()` → crea un objeto que 'acepta' cualquier llamada y devuelve otro `Mock`.
- `patch('modulo.objeto')` → sustituye temporalmente durante el test.
- `mock.assert_called_once_with(...)` → verifica cómo se llamó.

In [ ]:
# base de un caso REAL: una funcion que llama a una API externa.
# No queremos tocar la red real en los tests, asi que la simulamos.

from dataclasses import dataclass


@dataclass
class Configuracion:
    """Representa ajustes cargados desde un servicio externo."""
    tasa_impuesto: float = 0.16


def obtener_tasa_desde_api() -> float:
    """(Simulado) Consultaria una API real: lenta, externa, no fiable."""
    raise NotImplementedError("no llamar a la red real en tests")


def precio_final(precio: float, api: callable) -> float:
    """Aplica impuesto obtenido desde la API. Inyeccion del 'api' permite mockear."""
    tasa = api()
    return round(precio * (1 + tasa), 2)


print("Los mocks sustituiran a 'api' para no depender de la red.")

In [ ]:
# Usando unittest.mock.Mock directamente: creamos un 'api' falso.

from unittest.mock import Mock

api_mock = Mock(return_value=0.16)   # devolvera SIEMPRE 0.16

# Llamamos a nuestra funcion con el mock (no con la API real)
resultado = precio_final(1000, api_mock)
print("precio_final(1000, mock) =", resultado)  # 1000 * 1.16 = 1160.0

# Verificamos que la API fue llamada exactamente UNA vez
api_mock.assert_called_once_with()
print("La API (mock) fue llamada una vez -> OK")

# Podemos inspeccionar como se uso
print("Numero de llamadas:", api_mock.call_count)

In [ ]:
# Ahora integrado con unittest y mock.patch.
# patch temporalmente reemplaza un atributo y lo restaura al final.

from unittest.mock import patch


def obtener_precio_con_impuesto(precio: float, api: callable) -> float:
    """Funcion que DEPENDE de la API externa (no inyectada)."""
    tasa = api()
    return round(precio * (1 + tasa), 2)


# Queremos simular que la API devuelve una tasa especifica.
with patch("__main__.obtener_tasa_desde_api", return_value=0.10) as mock_tasa:
    resultado = obtener_precio_con_impuesto(1000, mock_tasa)

print("Con mock, la tasa fue 0.10 ->", resultado)  # 1100.0
mock_tasa.assert_called_once_with()
print("mock_tasa fue llamada una vez -> OK")

In [ ]:
# patch.object: parchea un ATRIBUTO de un objeto/clase.

from unittest.mock import patch


class ClienteDB:
    def conectar(self):
        return "conexion real a la BD (no queremos esto en tests)"

    def obtener_usuario(self, user_id):
        # Depende de una BD real
        self.conectar()
        return {"id": user_id, "nombre": "desde BD real"}


# Patchamos SOLO el metodo 'conectar' del objeto instanciado.
cliente = ClienteDB()

with patch.object(cliente, "conectar", return_value="conexion simulada") as mock_con:
    usuario = cliente.obtener_usuario(7)

print("usuario:", usuario)
mock_con.assert_called_once()
print("-> 'conectar' fue simulada, nunca se toco la BD real")

## 6. Cobertura de código

La **cobertura** mide *qué porcentaje de tu código* ejecutan los tests. Es útil para detectar **código muerto** o **ramas sin probar**.

### Cómo se calcula

```
                  lineas ejecutadas por los tests
   Cobertura % =  -----------------------------------  x 100
                  total de lineas ejecutables
```

GitHub, GitLab y la mayoría de CI pueden mostrar un badge de cobertura.

### pytest-cov

La herramienta estándar es **`pytest-cov`** (usa la librería `coverage.py` por debajo):

```bash
pip install pytest-cov
pytest --cov=mi_modulo --cov-report=term-missing
```

Salida típica:

```
Name           Stmts   Miss  Cover   Missing
mi_modulo.py      42      3    93%    17, 28-29
```

### ⚠️ No persigas el 100% ciegamente

- Cobertura **alta** no significa código **correcto** (puedes cubrir líneas sin probar la lógica).
- Los **branches** (ramas `if`) son más valiosos que las líneas.
- Enfócate en cubrir la **lógica crítica y los casos límite**, no en inflar el porcentaje.
- Un umbral razonable de proyecto es **~80-85%**, no 100%.

**Prioridad:** que las *decisiones* (ramas, excepciones) estén probadas vale más que un número de líneas perfecto.

In [ ]:
# Ilustracion de cobertura: probamos las RAMAS.
# Cobertura de ramas > cobertura de lineas.

def clasificar_precio(precio: float) -> str:
    """Clasifica un precio como barato, normal o caro."""
    if precio < 50:
        return "barato"
    elif precio < 500:
        return "normal"
    else:
        return "caro"


import unittest

class TestClasificar(unittest.TestCase):
    def test_barato(self):
        self.assertEqual(clasificar_precio(10), "barato")

    def test_normal(self):
        self.assertEqual(clasificar_precio(300), "normal")

    def test_caro(self):
        self.assertEqual(clasificar_precio(1000), "caro")


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestClasificar)
r = unittest.TextTestRunner(verbosity=2).run(suite)
print(f"\nCubrimos las 3 ramas (barato/normal/caro) -> {r.testsRun} tests OK")

## 7. Principios de calidad

Los tests no son solo protección: te obligan a escribir **código mejor diseñado**. Estos principios te ayudan a que el código sea **comprobable**.

### DRY - Don't Repeat Yourself

No repitas lógica. Extrae a funciones/constantes. En tests, usa `fixtures` y helpers.

```python
# MAL: repites la regla en 3 sitios
desc1 = total * 0.1
desc2 = total2 * 0.1

# BIEN: una sola fuente de verdad
TASA_DESC = 0.1
def descuento(t): return t * TASA_DESC
```

### KISS - Keep It Simple, Stupid

La solución más simple que funciona es la mejor. Código simple = más fácil de probar.

### SOLID (aplicado en Python)

| Letra | Principio | En Python |
|-------|-----------|-----------|
| S | Single Responsibility | cada función/clase hace UNA cosa |
| O | Open/Closed | abierto a extensión, cerrado a modificación (interfaces/Protocol) |
| L | Liskov | subclases no rompen el contrato de la base |
| I | Interface Segregation | no obligues a métodos que no se usan |
| D | Dependency Inversion | depende de abstracciones, no de implementaciones concretas |

### Diseño comprobable e inyección de dependencias

**Inyección de dependencias**: en vez de que tu función *cree* sus dependencias internas, **recíbelas** como parámetros. Así puedes pasar mocks en los tests.

```python
# MAL: dependencia oculta y no comprobable
def guardar():
    db = conexion_global()   # no puedes simularla
    db.save(...)

# BIEN: dependencia inyectada y comprobable
def guardar(db):             # inyected -> facil de mockear
    db.save(...)
```

### Cheatsheet de 'qué NO hacer'

| Evita | Mejor práctica |
|-------|----------------|
| Testear código que llama a la red/BD directamente | Inyector la dependencia y usar mocks |
| `assert` sin mensaje en unittest | Añade mensaje descriptivo al fallar |
| Un test gigante que prueba todo | Un test = un comportamiento |
| Tests que dependen de otros (orden) | Cada test debe ser independiente |
| Comparar flotantes con `==` | Usa `assertAlmostEqual` / tolerancia |
| Código con efectos secundarios globales | Funciones puras y estado inyectado |
| Ocultar errores con `try/except: pass` | Deja que el test vea el error |


## Tabla de referencia rápida

### unittest vs pytest

| Característica | unittest (stdlib) | pytest |
|----------------|-------------------|--------|
| Instalación | Ninguna (incluido) | `pip install pytest` |
| Estructura | Clases `TestCase` | Funciones `test_*` |
| Aserciones | `self.assertEqual`, etc. | `assert` de Python |
| Preparación | `setUp`/`tearDown` | `fixtures` con `yield` |
| Casos múltiples | Manual / subTest | `@pytest.mark.parametrize` |
| Saltar tests | `@unittest.skip` | `@pytest.mark.skip` |
| Fallo esperado | `@unittest.expectedFailure` | `@pytest.mark.xfail` |
| Reporte | Básico | Rico (colores, diagnóstico) |
| Plugins/eco | Limitado | Enorme ecosistema |
| ¿Cuándo? | Entornos sin internet | Estándar industrial |

### Aserciones equivalente

| Concepto | unittest | pytest |
|----------|----------|--------|
| Igualdad | `assertEqual(a,b)` | `assert a == b` |
| Verdadero | `assertTrue(x)` | `assert x` |
| Excepción | `assertRaises(E)` | `pytest.raises(E)` |
| Aproximado | `assertAlmostEqual(a,b)` | `abs(a-b) < tol` o `pytest.approx` |
| Contiene | `assertIn(a,b)` | `assert a in b` |
| None | `assertIsNone(x)` | `assert x is None` |

## Ejercicios

Aplica lo aprendido. Empieza con los **guiados** y luego ataca el **independiente**.

---

### Ejercicio 1 (guiado): test de una calculadora con unittest

Escribe una clase `TestCalculadora(unittest.TestCase)` que pruebe estas operaciones de una función `sumar(a, b)`:

- `sumar(2, 3) == 5`
- `sumar(-1, 1) == 0`
- `sumar(0, 0) == 0`

Completa el código y luego ejecútalo con `unittest.TextTestRunner`.

---

### Ejercicio 2 (guiado): fixture con pytest

Crea un `fixture` llamado `numeros` que devuelva `[1, 2, 3, 4, 5]`, y un test `test_suma` que verifique que la suma del fixture es `15`. Usa `yield` y nota que la preparación se repite en cada test.

---

### Ejercicio 3 (guiado): mock de una API

Usa `unittest.mock.patch` para simular la función `obtener_precio_dolar()` (que en producción consultaría una API) devolviendo `18.5`, y verifica que `convertir_a_pesos(100) == 1850.0`.

---

### Ejercicio 4 (independiente): suíte completa

Escribe una calculadora y sus tests **con pytest** (si está disponible) y/o **unittest** (siempre funciona):

1. Funciones: `sumar`, `restar`, `multiplicar`, `dividir` (dividir por cero debe lanzar `ValueError`).
2. Escribe tests para **casos normales**, **casos límite** (cero, negativos) y **errores**.
3. Usa `parametrize` para los 4 casos de `sumar` en una línea.
4. Agrega al menos un `skip` y un `xfail`.

**Pista:** empieza en RED (tests primero), implementa en GREEN, refactoriza.

In [ ]:
# ===== SOLUCION EJERCICIO 1 =====

def sumar(a, b):
    return a + b

import unittest

class TestCalculadora(unittest.TestCase):
    def test_suma_positivos(self):
        self.assertEqual(sumar(2, 3), 5)

    def test_suma_negativo_y_positivo(self):
        self.assertEqual(sumar(-1, 1), 0)

    def test_suma_ceros(self):
        self.assertEqual(sumar(0, 0), 0)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCalculadora)
r = unittest.TextTestRunner(verbosity=2).run(suite)
print(f"\nEjercicio 1: {r.testsRun} tests, {len(r.failures)} fallos")

In [ ]:
# ===== SOLUCION EJERCICIO 2 (fixture con yield) =====

def numeros():
    print("[setup] preparando lista")
    yield [1, 2, 3, 4, 5]
    print("[teardown] listo")


def test_suma():
    gen = numeros()
    datos = next(gen)
    assert sum(datos) == 15, f"esperaba 15, obtuve {sum(datos)}"
    try:
        next(gen)
    except StopIteration:
        pass
    print("-> suma == 15, OK")


test_suma()

In [ ]:
# ===== SOLUCION EJERCICIO 3 (mock de API) =====

from unittest.mock import patch

def obtener_precio_dolar():
    """En produccion consultaria una API real. Aqui simulamos con Mock."""
    raise NotImplementedError("no llamar red real")

def convertir_a_pesos(dolares, api):
    return dolares * api()


with patch("__main__.obtener_precio_dolar", return_value=18.5) as mock_api:
    resultado = convertir_a_pesos(100, mock_api)

assert resultado == 1850.0, f"esperaba 1850.0, obtuve {resultado}"
mock_api.assert_called_once_with()
print("Ejercicio 3: 100 USD ->", resultado, "pesos (API simulada)")
print("OK: la API fue llamada una vez y no se toco la red.")

In [ ]:
# ===== SOLUCION EJERCICIO 4 (independiente) con unittest SIEMPRE funcional =====
# La misma logica serviria con pytest (assert simple + parametrize).

def sumar(a, b):
    return a + b

def restar(a, b):
    return a - b

def multiplicar(a, b):
    return a * b

def dividir(a, b):
    if b == 0:
        raise ValueError("division por cero")
    return a / b

import unittest

class TestCalculadoraCompleta(unittest.TestCase):
    # suma parametrizada simulando @parametrize
    def test_suma_varios_casos(self):
        casos = [(2, 3, 5), (-1, 1, 0), (0, 0, 0), (10, -10, 0)]
        for a, b, esp in casos:
            with self.subTest(a=a, b=b):
                self.assertEqual(sumar(a, b), esp)

    def test_resta(self):
        self.assertEqual(restar(10, 4), 6)
        self.assertEqual(restar(4, 10), -6)

    def test_multiplicacion(self):
        self.assertEqual(multiplicar(3, 4), 12)
        self.assertEqual(multiplicar(0, 5), 0)

    def test_division(self):
        self.assertEqual(dividir(10, 2), 5.0)

    def test_division_por_cero(self):
        with self.assertRaises(ValueError):
            dividir(10, 0)

    @unittest.skip("Funcionalidad futura: potencia")
    def test_potencia(self):
        self.assertEqual(2 ** 8, 256)  # nunca se ejecuta

    @unittest.expectedFailure
    def test_division_enteros_esperado_fallar(self):
        # Demostracion de expectedFailure: se marca como failure esperado
        self.assertEqual(dividir(9, 2), 4)  # en realidad es 4.5 -> falla (esperado)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCalculadoraCompleta)
r = unittest.TextTestRunner(verbosity=2).run(suite)
print(f"\nEjercicio 4: {r.testsRun} tests, {len(r.failures)} fallos, {len(r.skipped)} omitidos")

### Verificación final del ejercicio 4 con pytest (si está instalado)

La misma suite en estilo pytest sería mucho más concisa. Si pytest está disponible, corre desde terminal:

```bash
pytest -q test_calculadora.py
```

Con `parametrize`, los 4 casos de suma serían:

```python
import pytest

@pytest.mark.parametrize("a,b,esperado", [(2,3,5), (-1,1,0), (0,0,0), (10,-10,0)])
def test_sumar(a, b, esperado):
    assert sumar(a, b) == esperado
```

## Resumen

- **Testear** es inversión: previene regresiones, documenta el comportamiento y da confianza para refactorizar.
- **unittest** está en la stdlib → **siempre funciona**; ideal para entornos controlados.
- **pytest** es el **estándar moderno**: `assert` simple, `fixtures` con `yield`, `parametrize` y `markers` (`skip`, `xfail`).
- **TDD** es Red → Green → Refactor: el test (contrato) se escribe primero, se ve fallar, se implementa el mínimo, y se limpia sin romper el verde.
- **Mocks** (`Mock`, `patch`, `patch.object`) aíslan dependencias externas (red/BD) para tests rápidos y fiables.
- **Cobertura** mide qué se probó, pero **no persigas el 100% ciego**: prioriza ramas y lógica crítica (~80-85% razonable).
- **Calidad**: DRY, KISS, SOLID; diseña código **comprobable** con **inyección de dependencias**.

### Flujo recomendado en tus proyectos

```
Escribir test (RED)
        |
        v
Implementar minimo (GREEN)
        |
        v
Refactorizar (sin romper verde)
        |
        v
Correr suite completa (CI + local)
        |
        v
Revisar cobertura (sin obsesionarse)
```

> **Reflexión final:** un proyecto sin tests *funciona hoy*. Un proyecto con tests *sigue funcionando mañana*. La diferencia es la confianza para evolucionar.